In [ ]:
# ============================================================================
# CELL 1: Import Libraries
# ============================================================================
import re
import nltk
from nltk import ngrams
from collections import Counter, defaultdict

print("Libraries imported successfully.")


Libraries imported successfully.


In [ ]:
# ============================================================================
# CELL 2: Configuration
# ============================================================================
DATA_FILE_PATH = 'quran_syl.txt'

print(f"Data file path set to: {DATA_FILE_PATH}")


Data file path set to: quran_syl.txt


In [ ]:
# ============================================================================
# CELL 3: Define Surah Structure - Verse Counts
# ============================================================================
# Ayah counts for each of the 114 Surahs (Hafs narration standard)
VERSES_PER_SURAH = [
    7, 286, 200, 176, 120, 165, 206, 75, 129, 109, 123, 111, 43, 52, 99, 128, 111, 110, 98, 135,
    112, 78, 118, 64, 77, 227, 93, 88, 69, 60, 34, 30, 73, 54, 45, 83, 182, 88, 75, 85,
    54, 53, 89, 59, 37, 35, 38, 29, 18, 45, 60, 49, 62, 55, 78, 96, 29, 22, 24, 13,
    14, 11, 11, 18, 12, 12, 30, 52, 52, 44, 28, 28, 20, 56, 40, 31, 50, 40, 46, 42, 29,
    19, 36, 25, 22, 17, 19, 26, 30, 20, 15, 21, 11, 8, 8, 19, 5, 8, 8, 11, 11, 8, 3,
    9, 5, 4, 7, 3, 6, 3, 5, 4, 5, 6
]

print(f"Total Surahs defined: {len(VERSES_PER_SURAH)}")
print(f"Total verses in Quran: {sum(VERSES_PER_SURAH)}")


Total Surahs defined: 114
Total verses in Quran: 6236


In [ ]:
# ============================================================================
# CELL 4: Define Surah Structure - Chapter Names
# ============================================================================
CHAPTER_NAMES = [
    "Al-Fatiha", "Al-Baqarah", "Al-Imran", "An-Nisa", "Al-Ma'idah", "Al-An'am", "Al-A'raf", "Al-Anfal", "At-Tawbah", "Yunus",
    "Hud", "Yusuf", "Ar-Ra'd", "Ibrahim", "Al-Hijr", "An-Nahl", "Al-Isra", "Al-Kahf", "Maryam", "Ta-Ha",
    "Al-Anbiya", "Al-Hajj", "Al-Mu'minun", "An-Nur", "Al-Furqan", "Ash-Shu'ara", "An-Naml", "Al-Qasas", "Al-Ankabut", "Ar-Rum",
    "Luqman", "As-Sajdah", "Al-Ahzab", "Saba", "Fatir", "Ya-Sin", "As-Saffat", "Sad", "Az-Zumar", "Ghafir",
    "Fussilat", "Ash-Shura", "Az-Zukhruf", "Ad-Dukhan", "Al-Jathiyah", "Al-Ahqaf", "Muhammad", "Al-Fath", "Al-Hujurat", "Qaf",
    "Adh-Dhariyat", "At-Tur", "An-Najm", "Al-Qamar", "Ar-Rahman", "Al-Waqi'ah", "Al-Hadid", "Al-Mujadila", "Al-Hashr", "Al-Mumtahanah", "As-Saff",
    "Al-Jumu'ah", "Al-Munafiqun", "At-Taghabun", "At-Talaq", "At-Tahrim", "Al-Mulk", "Al-Qalam", "Al-Haqqah", "Al-Ma'arij", "Nuh",
    "Al-Jinn", "Al-Muzzammil", "Al-Muddaththir", "Al-Qiyamah", "Al-Insan", "Al-Mursalat", "An-Naba", "An-Nazi'at", "Abasa", "At-Takwir",
    "Al-Infitar", "Al-Mutaffifin", "Al-Inshiqaq", "Al-Buruj", "At-Tariq", "Al-A'la", "Al-Ghashiyah", "Al-Fajr", "Al-Balad", "Ash-Shams",
    "Al-Layl", "Ad-Duha", "Ash-Sharh", "At-Tin", "Al-Alaq", "Al-Qadr", "Al-Bayyinah", "Az-Zalzalah", "Al-Adiyat", "Al-Qari'ah",
    "At-Takathur", "Al-Asr", "Al-Humazah", "Al-Fil", "Quraysh", "Al-Ma'un", "Al-Kawthar", "Al-Kafirun", "An-Nasr", "Al-Masad",
    "Al-Ikhlas", "Al-Falaq", "An-Nas"
]

print(f"Chapter names loaded: {len(CHAPTER_NAMES)}")


Chapter names loaded: 114


In [ ]:
# ============================================================================
# CELL 5: Helper Function - Find Verse Location
# ============================================================================
def find_verse_location(line_number):
    """
    Determines which Surah and Ayah a line belongs to based on line index.
    """
    running_total = 0
    for chapter_idx, verse_count in enumerate(VERSES_PER_SURAH):
        if line_number < running_total + verse_count:
            verse_number = (line_number - running_total) + 1
            return CHAPTER_NAMES[chapter_idx], verse_number
        running_total += verse_count
    return "Unknown", 0

print("Helper function 'find_verse_location' defined.")


Helper function 'find_verse_location' defined.


In [ ]:
# ============================================================================
# CELL 6: Data Loading Function
# ============================================================================
def read_and_process_data(file_path):
    """
    Loads the Quran syllable file and maps each syllable to its location.
    """
    syllable_list = []
    location_info = []

    try:
        with open(file_path, 'r', encoding='utf-8') as file:
            text_lines = file.readlines()

        print(f"Successfully loaded file. Line count: {len(text_lines)}")

        for line_idx, text_line in enumerate(text_lines):
            cleaned_line = text_line.strip()
            if not cleaned_line:
                continue

            chapter, verse = find_verse_location(line_idx)
            line_syllables = [syl for syl in re.split(r'[\s\-]+', cleaned_line) if syl]

            for syllable in line_syllables:
                syllable_list.append(syllable)
                location_info.append((chapter, verse))

        return syllable_list, location_info, text_lines

    except FileNotFoundError:
        print(f"Error: Cannot find file at {file_path}. Please verify the path.")
        return [], [], []

print("Data loading function 'read_and_process_data' defined.")

# syllable_list:  ["Al", "ham", "du"]
# location_info:  [("Al-Fatiha", 1), ("Al-Fatiha", 1), ("Al-Fatiha", 1)]


Data loading function 'read_and_process_data' defined.


In [ ]:
# ============================================================================
# CELL 7: Load and Parse Data
# ============================================================================
all_syllables, syllable_locations, raw_text_lines = read_and_process_data(DATA_FILE_PATH)
print(f"Syllables extracted: {len(all_syllables)}")
print(f"Total lines processed: {len(raw_text_lines)}")


Successfully loaded file. Line count: 6236
Syllables extracted: 210545
Total lines processed: 6236


In [ ]:
# ============================================================================
# CELL 8: Analysis Function - Common N-grams
# ============================================================================
def find_common_ngrams(gram_size, display_count=10):
    """
    Task 1: Identifies the most frequently occurring N-grams and their primary Surahs.
    """
    print(f"\n>>> Most Common {gram_size}-gram Patterns <<<")

    ngram_list = list(ngrams(all_syllables, gram_size))
    frequency_counts = Counter(ngram_list)

    for pattern_tuple, occurrence_count in frequency_counts.most_common(display_count):
        pattern_string = "-".join(pattern_tuple)
        pattern_positions = [idx for idx, gram in enumerate(ngram_list) if gram == pattern_tuple]
        surah_list = [syllable_locations[pos][0] for pos in pattern_positions]
        top_surahs = Counter(surah_list).most_common(3)
        surah_display = ", ".join([f"{chapter}({count})" for chapter, count in top_surahs])

        print(f"Sequence: {pattern_string} | Occurrences: {occurrence_count} | Main Surahs: {surah_display}")

print("Analysis function 'find_common_ngrams' defined.")

# ("Al","ham","du")
# ("ham","du","li")
# ("du","li","llah")


Analysis function 'find_common_ngrams' defined.


In [ ]:
# ============================================================================
# CELL 9: Analysis Function - Longest N-gram
# ============================================================================
def find_longest_ngram(gram_size):
    """
    Task 2: Identifies N-grams with maximum character length and their location.
    """
    ngram_list = list(ngrams(all_syllables, gram_size))
    if not ngram_list:
        return

    max_length_pattern = max(ngram_list, key=lambda gram: len("".join(gram)))
    position = ngram_list.index(max_length_pattern)
    surah_verse = syllable_locations[position]

    print(f"\n>>> Longest {gram_size}-gram Pattern <<<")
    print(f"Sequence: {'-'.join(max_length_pattern)}")
    print(f"Character Count: {len(''.join(max_length_pattern))}")
    print(f"Found in: {surah_verse[0]}, Verse {surah_verse[1]}")

print("Analysis function 'find_longest_ngram' defined.")

# ("wa","rah","ma") → "warahma" (7 chars)
# ("bi","smil","lah") → "bismillah" (10 chars) ✅


Analysis function 'find_longest_ngram' defined.


In [ ]:
# ============================================================================
# CELL 10: Analysis Function - Number 19 Pattern
# ============================================================================
def explore_number_19_pattern():
    """
    Task 3: Investigates the Number 19 phenomenon in the Quran.
    """
    print("\n>>> Investigating the Number 19 Mystery <<<")

    chapter_syllable_totals = defaultdict(int)
    verses_divisible_by_19 = []

    for line_idx, text_line in enumerate(raw_text_lines):
        chapter, verse = find_verse_location(line_idx)
        line_syllables = [syl for syl in re.split(r'[\s\-]+', text_line.strip()) if syl]
        syllable_count = len(line_syllables)
        chapter_syllable_totals[chapter] += syllable_count

        if syllable_count > 0 and syllable_count % 19 == 0:
            verses_divisible_by_19.append((chapter, verse, syllable_count))

    print("\n[Part 1] Chapters with syllable counts divisible by 19:")
    chapters_found = False
    for chapter, total in chapter_syllable_totals.items():
        if total > 0 and total % 19 == 0:
            print(f" - {chapter}: {total} syllables")
            chapters_found = True
    if not chapters_found:
        print(" No chapters found.")

    print(f"\n[Part 2] Verses divisible by 19: {len(verses_divisible_by_19)}")
    print(" Example matches:")
    for match in verses_divisible_by_19[:3]:
        print(f" - {match[0]} Verse {match[1]} ({match[2]} syllables)")

    print("\n[Part 3] Analysis of 19-syllable sequences:")
    nineteen_grams = list(ngrams(all_syllables, 19))
    nineteen_gram_counts = Counter(nineteen_grams)
    repeated_patterns = {pattern: count for pattern, count in nineteen_gram_counts.items() if count > 1}

    print(f" Total distinct 19-grams: {len(nineteen_gram_counts)}")
    print(f" Repeated 19-grams discovered: {len(repeated_patterns)}")

    if repeated_patterns:
        most_frequent = Counter(repeated_patterns).most_common(1)[0]
        print(f" Most repeated 19-gram (appears {most_frequent[1]} times):")
        print(f" Pattern: {'-'.join(most_frequent[0])[:100]}...")

print("Analysis function 'explore_number_19_pattern' defined.")


Analysis function 'explore_number_19_pattern' defined.


In [ ]:
# ============================================================================
# CELL 11: Execute Analysis - Common N-grams (1-gram)
# ============================================================================
if all_syllables:
    find_common_ngrams(1)
else:
    print("Data not loaded. Please check previous cells.")



>>> Most Common 1-gram Patterns <<<
Sequence: وَ | Occurrences: 8776 | Main Surahs: Al-Baqarah(694), An-Nisa(477), Al-Imran(387)
Sequence: لَ | Occurrences: 7274 | Main Surahs: Al-Baqarah(619), Al-A'raf(366), An-Nisa(306)
Sequence: لَاْ | Occurrences: 6680 | Main Surahs: Al-Baqarah(600), An-Nisa(437), Al-Imran(363)
Sequence: نَ | Occurrences: 6650 | Main Surahs: Al-Baqarah(478), An-Nisa(419), Al-A'raf(285)
Sequence: مَاْ | Occurrences: 3742 | Main Surahs: Al-Baqarah(305), An-Nisa(191), Al-An'am(165)
Sequence: ھُ | Occurrences: 3663 | Main Surahs: Al-Baqarah(312), An-Nisa(175), Al-Imran(167)
Sequence: عَ | Occurrences: 3603 | Main Surahs: Al-Baqarah(289), An-Nisa(188), Al-An'am(147)
Sequence: فَ | Occurrences: 3441 | Main Surahs: Al-Baqarah(295), An-Nisa(209), Al-Imran(138)
Sequence: نَاْ | Occurrences: 3268 | Main Surahs: Al-Baqarah(224), Al-A'raf(217), An-Nisa(122)
Sequence: تَ | Occurrences: 3196 | Main Surahs: Al-Baqarah(297), An-Nisa(162), Al-Imran(154)


In [ ]:
# ============================================================================
# CELL 12: Execute Analysis - Common N-grams (3-gram)
# ============================================================================
if all_syllables:
    find_common_ngrams(3)



>>> Most Common 3-gram Patterns <<<
Sequence: لَ-ذِيْ-نَ | Occurrences: 1018 | Main Surahs: Al-Baqarah(81), Al-Imran(61), An-Nisa(59)
Sequence: ذَ-لِ-كَ | Occurrences: 401 | Main Surahs: Al-Baqarah(27), Al-Ma'idah(18), Al-An'am(18)
Sequence: نَلْ-لَاْ-ھَ | Occurrences: 380 | Main Surahs: Al-Baqarah(51), An-Nisa(34), Al-Hajj(31)
Sequence: ءِنْ-نَلْ-لَاْ | Occurrences: 263 | Main Surahs: Al-Baqarah(33), An-Nisa(30), Al-Imran(23)
Sequence: ذِيْ-نَ-ءَاْ | Occurrences: 253 | Main Surahs: Al-Baqarah(28), Al-Ma'idah(24), An-Nisa(17)
Sequence: لَاْ-ھِ-وَ | Occurrences: 246 | Main Surahs: An-Nisa(31), Al-Baqarah(26), Al-Imran(26)
Sequence: نَ-ءَاْ-مَ | Occurrences: 242 | Main Surahs: Al-Baqarah(26), Al-Ma'idah(24), An-Nisa(17)
Sequence: وَلْ-لَاْ-ھُ | Occurrences: 241 | Main Surahs: Al-Baqarah(40), Al-Imran(38), At-Tawbah(23)
Sequence: وَلْ-لَ-ذِيْ | Occurrences: 237 | Main Surahs: Al-Baqarah(16), Al-An'am(15), Al-A'raf(14)
Sequence: ءَاْ-مَ-نُوْ | Occurrences: 234 | Main Surahs: Al-Baqarah(25

In [ ]:
# ============================================================================
# CELL 13: Execute Analysis - Common N-grams (4-gram)
# ============================================================================
if all_syllables:
    find_common_ngrams(4)



>>> Most Common 4-gram Patterns <<<
Sequence: ءِنْ-نَلْ-لَاْ-ھَ | Occurrences: 258 | Main Surahs: Al-Baqarah(31), An-Nisa(30), Al-Imran(21)
Sequence: لَ-ذِيْ-نَ-ءَاْ | Occurrences: 253 | Main Surahs: Al-Baqarah(28), Al-Ma'idah(24), An-Nisa(17)
Sequence: ذِيْ-نَ-ءَاْ-مَ | Occurrences: 242 | Main Surahs: Al-Baqarah(26), Al-Ma'idah(24), An-Nisa(17)
Sequence: نَ-ءَاْ-مَ-نُوْ | Occurrences: 218 | Main Surahs: Al-Baqarah(23), Al-Ma'idah(19), An-Nisa(17)
Sequence: لَ-ذِيْ-نَ-كَ | Occurrences: 176 | Main Surahs: Al-Imran(15), Al-Anfal(11), Al-Baqarah(9)
Sequence: ءُوْ-لَ-ءِ-كَ | Occurrences: 173 | Main Surahs: Al-Baqarah(24), Al-Imran(13), An-Nisa(12)
Sequence: ذِيْ-نَ-كَ-فَ | Occurrences: 171 | Main Surahs: Al-Imran(15), Al-Anfal(11), Al-Baqarah(9)
Sequence: نَ-كَ-فَ-رُوْ | Occurrences: 164 | Main Surahs: Al-Imran(14), Al-Baqarah(9), An-Nisa(9)
Sequence: نَلْ-لَ-ذِيْ-نَ | Occurrences: 163 | Main Surahs: Al-Imran(18), Al-Ma'idah(13), An-Nisa(12)
Sequence: وَلْ-لَ-ذِيْ-نَ | Occurrences: 157 | 

In [ ]:
# ============================================================================
# CELL 14: Execute Analysis - Common N-grams (5-gram)
# ============================================================================
if all_syllables:
    find_common_ngrams(5)



>>> Most Common 5-gram Patterns <<<
Sequence: لَ-ذِيْ-نَ-ءَاْ-مَ | Occurrences: 242 | Main Surahs: Al-Baqarah(26), Al-Ma'idah(24), An-Nisa(17)
Sequence: ذِيْ-نَ-ءَاْ-مَ-نُوْ | Occurrences: 218 | Main Surahs: Al-Baqarah(23), Al-Ma'idah(19), An-Nisa(17)
Sequence: لَ-ذِيْ-نَ-كَ-فَ | Occurrences: 171 | Main Surahs: Al-Imran(15), Al-Anfal(11), Al-Baqarah(9)
Sequence: ذِيْ-نَ-كَ-فَ-رُوْ | Occurrences: 163 | Main Surahs: Al-Imran(14), An-Nisa(9), Al-Ma'idah(9)
Sequence: سَ-مَاْ-وَاْ-تِ-وَلْ | Occurrences: 118 | Main Surahs: Al-An'am(7), Ar-Rum(5), Az-Zumar(5)
Sequence: مَاْ-وَاْ-تِ-وَلْ-ءَرْ | Occurrences: 118 | Main Surahs: Al-An'am(7), Ar-Rum(5), Az-Zumar(5)
Sequence: وَاْ-تِ-وَلْ-ءَرْ-ضِ | Occurrences: 93 | Main Surahs: Al-Baqarah(5), Al-Imran(5), Al-Ma'idah(4)
Sequence: يَاْ-ءَيْ-يُ-ھَلْ-لَ | Occurrences: 93 | Main Surahs: Al-Ma'idah(16), Al-Baqarah(11), An-Nisa(10)
Sequence: ءَيْ-يُ-ھَلْ-لَ-ذِيْ | Occurrences: 93 | Main Surahs: Al-Ma'idah(16), Al-Baqarah(11), An-Nisa(10)
Sequence: يُ-ھَ

In [ ]:
# ============================================================================
# CELL 15: Execute Analysis - Longest N-grams (1 to 5)
# ============================================================================
if all_syllables:
    for n_size in range(1, 6):
        find_longest_ngram(n_size)



>>> Longest 1-gram Pattern <<<
Sequence: حِيْمْ
Character Count: 6
Found in: Al-Fatiha, Verse 1

>>> Longest 2-gram Pattern <<<
Sequence: ضَاْلْ-لِيْنْ
Character Count: 12
Found in: Al-Fatiha, Verse 7

>>> Longest 3-gram Pattern <<<
Sequence: لَاْمْ-مِيْمْ-صَاْدْ
Character Count: 18
Found in: Al-A'raf, Verse 1

>>> Longest 4-gram Pattern <<<
Sequence: مِيْمْ-عَيْنْ-سِيْنْ-قَاْفْ
Character Count: 24
Found in: Ash-Shura, Verse 1

>>> Longest 5-gram Pattern <<<
Sequence: حِيْطْ-حَاْ-مِيْمْ-عَيْنْ-سِيْنْ
Character Count: 28
Found in: Fussilat, Verse 54


In [ ]:
# ============================================================================
# CELL 16: Execute Analysis - Number 19 Pattern
# ============================================================================
if all_syllables:
    explore_number_19_pattern()



>>> Investigating the Number 19 Mystery <<<

[Part 1] Chapters with syllable counts divisible by 19:
 - Al-An'am: 8246 syllables
 - Al-Ankabut: 2717 syllables
 - Al-Jathiyah: 1311 syllables
 - Muhammad: 1558 syllables
 - Al-A'la: 190 syllables
 - Al-Balad: 209 syllables
 - Ad-Duha: 114 syllables
 - At-Takathur: 76 syllables

[Part 2] Verses divisible by 19: 253
 Example matches:
 - Al-Baqarah Verse 19 (57 syllables)
 - Al-Baqarah Verse 28 (38 syllables)
 - Al-Baqarah Verse 30 (76 syllables)

[Part 3] Analysis of 19-syllable sequences:
 Total distinct 19-grams: 207502
 Repeated 19-grams discovered: 2531
 Most repeated 19-gram (appears 8 times):
 Pattern: قَ-لَ-يَاْ-قَوْ-مِعْ-بُ-دُلْ-لَاْ-ھَ-مَاْ-لَ-كُمْ-مِنْ-ءِ-لَ-ھِنْ-غَيْ-رُ-ھُ...


In [ ]:
# ============================================================================
# CELL 17: Summary Statistics
# ============================================================================
if all_syllables:
    print("\n" + "="*70)
    print("ANALYSIS COMPLETE - SUMMARY")
    print("="*70)
    print(f"Total syllables analyzed: {len(all_syllables)}")
    print(f"Total verses processed: {len(raw_text_lines)}")
    print(f"Total Surahs: {len(CHAPTER_NAMES)}")
    print(f"Unique syllables: {len(set(all_syllables))}")
    print("="*70)



ANALYSIS COMPLETE - SUMMARY
Total syllables analyzed: 210545
Total verses processed: 6236
Total Surahs: 114
Unique syllables: 2097
